# 05 Chronological Split RESET
Creates train/validation/test splits using actual F3 and F4 warning starts. Does not drop rows due to missing feature values.



In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print('Python:', sys.executable)



In [ ]:
current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == 'notebooks' else current_dir
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
SPLIT_DATA_DIR = PROCESSED_DATA_DIR / 'splits'
OUTPUT_TABLES_DIR = PROJECT_ROOT / 'outputs' / 'tables'
OUTPUT_FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
for p in [SPLIT_DATA_DIR, OUTPUT_TABLES_DIR, OUTPUT_FIGURES_DIR]: p.mkdir(parents=True, exist_ok=True)
model_ready_path = PROCESSED_DATA_DIR / 'metropt3_model_ready_12h.csv'



In [ ]:
if not model_ready_path.exists(): raise FileNotFoundError('Run Step 4 first.')
df = pd.read_csv(model_ready_path)
timestamp_col = 'timestamp' if 'timestamp' in df.columns else [c for c in ['Timestamp','time','Time','datetime','Datetime'] if c in df.columns][0]
PRIMARY_TARGET='warning_12h'
if PRIMARY_TARGET not in df.columns: raise ValueError('warning_12h not found. Run Step 4.')
df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors='coerce')
df = df.dropna(subset=[timestamp_col]).sort_values(timestamp_col).reset_index(drop=True)
for c in ['warning_6h_event_id','warning_12h_event_id','warning_24h_event_id','failure_event_id']:
    if c in df.columns: df[c] = df[c].fillna('None').astype(str)
if df[timestamp_col].max() < pd.Timestamp('2020-07-01'): raise ValueError('Model-ready dataset is partial. Rerun Steps 2-4.')
print('Input shape:', df.shape, 'End:', df[timestamp_col].max())



In [ ]:
event_col='warning_12h_event_id'
events=[]
for event_id in ['F1','F2','F3','F4']:
    rows=df[df[event_col]==event_id]
    if len(rows)>0:
        events.append({'event_id':event_id,'warning_rows':len(rows),'first_warning_timestamp':rows[timestamp_col].min(),'last_warning_timestamp':rows[timestamp_col].max()})
event_ranges=pd.DataFrame(events)
event_ranges.to_csv(OUTPUT_TABLES_DIR / 'available_warning_event_ranges.csv', index=False)
print(df[event_col].value_counts().head(10))
event_ranges



In [ ]:
missing=[e for e in ['F1','F2','F3','F4'] if e not in event_ranges['event_id'].tolist()]
if missing: raise ValueError(f'Missing warning events {missing}. Rerun Steps 2-4.')
f3_start=event_ranges.loc[event_ranges['event_id']=='F3','first_warning_timestamp'].iloc[0]
f4_start=event_ranges.loc[event_ranges['event_id']=='F4','first_warning_timestamp'].iloc[0]
train_df=df[df[timestamp_col] < f3_start].copy()
val_df=df[(df[timestamp_col] >= f3_start) & (df[timestamp_col] < f4_start)].copy()
test_df=df[df[timestamp_col] >= f4_start].copy()
print('Train:', train_df.shape, train_df[timestamp_col].min(), train_df[timestamp_col].max())
print('Validation:', val_df.shape, val_df[timestamp_col].min(), val_df[timestamp_col].max())
print('Test:', test_df.shape, test_df[timestamp_col].min(), test_df[timestamp_col].max())
if min(len(train_df), len(val_df), len(test_df)) == 0: raise ValueError('A split is empty.')



In [ ]:
boundaries=pd.DataFrame({'split':['train','validation','test'],'start_time':[train_df[timestamp_col].min(),val_df[timestamp_col].min(),test_df[timestamp_col].min()],'end_time':[train_df[timestamp_col].max(),val_df[timestamp_col].max(),test_df[timestamp_col].max()],'main_warning_events':['F1, F2','F3','F4']})
boundaries.to_csv(OUTPUT_TABLES_DIR / 'split_date_boundaries.csv', index=False)
boundaries



In [ ]:
exclude=[timestamp_col,'failure_interval','failure_event_id','model_include','warning_6h','warning_12h','warning_24h','warning_6h_event_id','warning_12h_event_id','warning_24h_event_id']
candidates=[c for c in df.columns if c not in exclude]
feature_cols=[]
for c in candidates:
    tr=pd.to_numeric(train_df[c], errors='coerce')
    if tr.notna().sum() > 0:
        train_df[c]=tr; val_df[c]=pd.to_numeric(val_df[c], errors='coerce'); test_df[c]=pd.to_numeric(test_df[c], errors='coerce'); feature_cols.append(c)
pd.DataFrame({'feature_name':feature_cols}).to_csv(OUTPUT_TABLES_DIR / 'feature_columns_for_model.csv', index=False)
print('Feature columns:', len(feature_cols))



In [ ]:
train_df_clean=train_df.dropna(subset=[PRIMARY_TARGET]).copy()
val_df_clean=val_df.dropna(subset=[PRIMARY_TARGET]).copy()
test_df_clean=test_df.dropna(subset=[PRIMARY_TARGET]).copy()
X_train=train_df_clean[feature_cols].copy(); y_train=train_df_clean[PRIMARY_TARGET].astype(int)
X_val=val_df_clean[feature_cols].copy(); y_val=val_df_clean[PRIMARY_TARGET].astype(int)
X_test=test_df_clean[feature_cols].copy(); y_test=test_df_clean[PRIMARY_TARGET].astype(int)
print('X_train:', X_train.shape, 'X_val:', X_val.shape, 'X_test:', X_test.shape)
if len(X_val)==0: raise ValueError('Validation is empty. Do not continue.')



In [ ]:
def dist(data, name):
    total=len(data); warn=int(data[PRIMARY_TARGET].sum()); normal=total-warn
    ev=', '.join(sorted([x for x in data[event_col].unique() if x!='None'])) or 'None'
    return {'split':name,'normal_count':normal,'warning_count':warn,'total_rows':total,'warning_percentage':warn/total*100,'warning_event_ids':ev}
split_summary=pd.DataFrame([dist(train_df_clean,'train'), dist(val_df_clean,'validation'), dist(test_df_clean,'test')])
split_summary.to_csv(OUTPUT_TABLES_DIR / 'split_class_distribution.csv', index=False)
split_summary.to_csv(OUTPUT_TABLES_DIR / 'split_event_summary.csv', index=False)
split_summary



In [ ]:
x=np.arange(len(split_summary)); w=0.35
plt.figure(figsize=(10,6)); plt.bar(x-w/2, split_summary['normal_count'], w, label='Normal'); plt.bar(x+w/2, split_summary['warning_count'], w, label='Warning'); plt.xticks(x, split_summary['split']); plt.ylabel('Rows'); plt.title('Chronological split class distribution'); plt.legend(); plt.tight_layout(); plt.savefig(OUTPUT_FIGURES_DIR/'split_class_distribution.png', dpi=300); plt.show()



In [ ]:
plt.figure(figsize=(12,4))
for label, data, y in [('Train',train_df_clean,3),('Validation',val_df_clean,2),('Test',test_df_clean,1)]:
    plt.hlines(y, data[timestamp_col].min(), data[timestamp_col].max(), linewidth=10)
    plt.text(data[timestamp_col].min(), y+0.12, label)
for _, r in event_ranges.iterrows():
    plt.axvline(r.first_warning_timestamp, linestyle='--', alpha=.7); plt.text(r.first_warning_timestamp, 3.4, r.event_id, rotation=90)
plt.yticks([1,2,3], ['Test','Validation','Train']); plt.xlabel('Time'); plt.title('Chronological split timeline'); plt.tight_layout(); plt.savefig(OUTPUT_FIGURES_DIR/'chronological_split_timeline.png', dpi=300); plt.show()



In [ ]:
train_df_clean.to_csv(SPLIT_DATA_DIR/'train_12h.csv', index=False)
val_df_clean.to_csv(SPLIT_DATA_DIR/'validation_12h.csv', index=False)
test_df_clean.to_csv(SPLIT_DATA_DIR/'test_12h.csv', index=False)
X_train.to_csv(SPLIT_DATA_DIR/'X_train_12h.csv', index=False); y_train.to_frame(PRIMARY_TARGET).to_csv(SPLIT_DATA_DIR/'y_train_12h.csv', index=False)
X_val.to_csv(SPLIT_DATA_DIR/'X_validation_12h.csv', index=False); y_val.to_frame(PRIMARY_TARGET).to_csv(SPLIT_DATA_DIR/'y_validation_12h.csv', index=False)
X_test.to_csv(SPLIT_DATA_DIR/'X_test_12h.csv', index=False); y_test.to_frame(PRIMARY_TARGET).to_csv(SPLIT_DATA_DIR/'y_test_12h.csv', index=False)
chrono_summary=pd.DataFrame({'item':['Input rows','Feature columns','Train rows','Validation rows','Test rows','Train warnings','Validation warnings','Test warnings'], 'value':[df.shape[0],len(feature_cols),len(train_df_clean),len(val_df_clean),len(test_df_clean),int(y_train.sum()),int(y_val.sum()),int(y_test.sum())]})
chrono_summary.to_csv(OUTPUT_TABLES_DIR / 'chronological_split_summary.csv', index=False)
checklist=pd.DataFrame({'task':['F1-F4 found','Train non-empty','Validation non-empty','Test non-empty','Split files saved'], 'status':['Complete','Complete','Complete','Complete','Complete']})
checklist.to_csv(OUTPUT_TABLES_DIR / 'chronological_split_completion_checklist.csv', index=False)
print('Step 5 complete. Split files saved:', SPLIT_DATA_DIR)
chrono_summary
